# Fast masked-node DAS (ones-carry -> tens digit)

Speed changes vs. the previous notebook:
1. **Orthonormal basis computed once per step** with batched QR (the old `orthogonal` parametrization was re-evaluated on every access, i.e. every micro-batch).
2. **Basis / mask gradients accumulated on detached leaves**, then pushed back through QR once.
3. **Vectorized hooks**: one gather/einsum per layer instead of a Python loop per node.
4. **Minibatch steps** (e.g. 64 pairs per step) instead of full-dataset epochs.
5. **fp16 autocast** (T4 tensor cores) for the transformer; intervention math stays fp32.
6. Source activations kept on GPU. Timing is printed after the first steps.

Keep `subspace_dim` small (1-8). Dim 128 per node is almost never what DAS wants and makes everything slow.

In [ ]:
!pip install -q transformer_lens

In [ ]:
import time, math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformer_lens.model_bridge import TransformerBridge

model = TransformerBridge.boot_transformers("Qwen/Qwen3-1.7B", device="cuda")
model.eval()
model.requires_grad_(False)
device = next(model.parameters()).device
print("device:", device, "| layers:", model.cfg.n_layers, "| d_model:", model.cfg.d_model)

In [ ]:
template = "{} + {} = {}"

def create_dataset(n_samples=500, seed=0):
    rng = random.Random(seed)
    data, seen = [], set()
    while len(data) < n_samples:
        ah, bh = rng.randint(1, 8), rng.randint(1, 8)
        if ah + bh > 9:
            continue
        at = rng.randint(0, 8)
        bt = rng.randint(0, 8 - at)
        ao, bo = rng.randint(0, 9), rng.randint(0, 9)
        if ao + bo >= 10:
            continue
        co = rng.randint(1, 9)
        do = rng.randint(10 - co, 9)
        if co + do < 10:
            continue
        a = 100 * ah + 10 * at + ao
        b = 100 * bh + 10 * bt + bo
        c = 100 * ah + 10 * at + co
        d = 100 * bh + 10 * bt + do
        base_sum, source_sum = a + b, c + d
        if base_sum // 100 != source_sum // 100 or not 0 <= base_sum // 100 <= 9:
            continue
        base_prompt = template.format(a, b, base_sum // 100)
        source_prompt = template.format(c, d, source_sum // 100)
        if len(base_prompt) != len(source_prompt):
            continue
        base_target = (base_sum // 10) % 10
        source_target = (source_sum // 10) % 10
        if source_target != base_target + 1:
            continue
        pair = (base_prompt, source_prompt, base_target, source_target)
        if pair not in seen:
            data.append(pair)
            seen.add(pair)
    return data

def _prompt_tokens(model, prompts, device):
    toks = [model.to_tokens(p)[0] for p in prompts]
    L = toks[0].shape[0]
    if any(t.shape[0] != L for t in toks):
        raise ValueError("All prompts must tokenize to the same length")
    return torch.stack(toks).to(device)

def get_digit_ids(model):
    ids = []
    for i in range(10):
        t = model.to_tokens(str(i))
        if t.shape[1] != 1:
            raise ValueError(f"digit {i} tokenizes to {t.shape[1]} tokens")
        ids.append(t[0, -1].item())
    return ids

def check_tokenization(model, prompt="410 + 432 = 8"):
    toks = list(model.to_str_tokens(prompt))
    expected = ['4', '1', '0', ' +', ' ', '4', '3', '2', ' =', ' ', '8']
    if toks != expected:
        raise ValueError(f"Unexpected tokenization {toks}; expected {expected}.")
    completed = list(model.to_str_tokens(prompt + "4"))
    if completed[:-1] != toks or completed[-1] != "4":
        raise ValueError("Tokenizer does not preserve prefix when digit appended.")
    return toks

@torch.no_grad()
def _score_next_digit(model, prompts, target_digits, digit_ids, batch_size=64, amp=True):
    dev = next(model.parameters()).device
    targets = torch.as_tensor([digit_ids[int(d)] for d in target_digits], device=dev, dtype=torch.long)
    preds, losses = [], []
    for s in range(0, len(prompts), batch_size):
        e = min(s + batch_size, len(prompts))
        toks = _prompt_tokens(model, prompts[s:e], dev)
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
            logits = model.run_with_hooks(toks, return_type="logits")[:, -1, :].float()
        preds.extend(logits.argmax(-1).cpu().tolist())
        losses.extend(F.cross_entropy(logits, targets[s:e], reduction="none").cpu().tolist())
    return preds, losses

def create_filtered_dataset(model, digit_ids, n_samples=300, seed=0, score_batch_size=64,
                            max_candidates_factor=20, amp=True):
    cand, accepted, seen = 0, [], set()
    bh_all, sh_all, bl_all, sl_all = [], [], [], []
    attempt = 0
    while len(accepted) < n_samples:
        remaining = n_samples - len(accepted)
        batch = create_dataset(max(32, math.ceil(1.28 * remaining)), seed=seed + attempt)
        attempt += 1
        fresh = []
        for row in batch:
            key = (row[0], row[1])
            if key not in seen:
                seen.add(key); fresh.append(row)
        if not fresh:
            continue
        bp, sp = [r[0] for r in fresh], [r[1] for r in fresh]
        bt, st = [r[2] for r in fresh], [r[3] for r in fresh]
        bpred, bloss = _score_next_digit(model, bp, bt, digit_ids, score_batch_size, amp)
        spred, sloss = _score_next_digit(model, sp, st, digit_ids, score_batch_size, amp)
        for i, row in enumerate(fresh):
            b_ok = bpred[i] == digit_ids[int(row[2])]
            s_ok = spred[i] == digit_ids[int(row[3])]
            cand += 1
            bh_all.append(b_ok); sh_all.append(s_ok); bl_all.append(bloss[i]); sl_all.append(sloss[i])
            if b_ok and s_ok and len(accepted) < n_samples:
                accepted.append((row, bloss[i], sloss[i]))
        if cand >= max_candidates_factor * n_samples and len(accepted) < n_samples:
            raise RuntimeError(f"Only {len(accepted)}/{n_samples} kept after {cand} candidates.")
    print(f"candidates={cand} | base acc={np.mean(bh_all):.3f} CE={np.mean(bl_all):.3f} | "
          f"source acc={np.mean(sh_all):.3f} CE={np.mean(sl_all):.3f} | "
          f"both-correct={sum(a and b for a, b in zip(bh_all, sh_all))}/{cand}")
    return [r for r, _, _ in accepted]

In [ ]:
# Candidate nodes: (layer, position). Positions 7-10, first 70% of layers.
n_search_layers = max(1, int(0.7 * model.cfg.n_layers))
nodes = [(layer, pos) for layer in range(n_search_layers) for pos in range(7, 11)]
print(len(nodes), "candidate nodes")

In [ ]:
# ----------------------------------------------------------------------------
# Core: vectorized interchange intervention
# ----------------------------------------------------------------------------
def binary_concrete(logits, K=1.0, hard=True):
    soft = torch.sigmoid(K * logits)
    if not hard:
        return soft, soft
    hard_m = (logits > 0).to(logits.dtype)
    return hard_m - soft.detach() + soft, soft


class DASSubspaces(nn.Module):
    """Unconstrained W [n_nodes, d, k]; orthonormal basis Q via one batched QR per call."""
    def __init__(self, n_nodes, d_model, k=1):
        super().__init__()
        self.k = k
        self.W = nn.Parameter(torch.randn(n_nodes, d_model, k) / d_model ** 0.5)
    def basis(self):
        return torch.linalg.qr(self.W).Q          # [n, d, k], orthonormal columns


def prepare(model, data, nodes, gold_ids, cache_bs=64, amp=True):
    """Tokenize and cache SOURCE residual vectors for every node (kept on GPU, fp32)."""
    dev = next(model.parameters()).device
    gold = torch.as_tensor(gold_ids, dtype=torch.long, device=dev).view(-1)
    N = len(data)
    base_tok = _prompt_tokens(model, [r[0] for r in data], dev)
    src_tok = _prompt_tokens(model, [r[1] for r in data], dev)
    L = base_tok.shape[1]
    by_layer = {}
    for i, (layer, pos) in enumerate(nodes):
        by_layer.setdefault(layer, []).append((i, pos))
    layer_nodes = {l: torch.tensor([i for i, _ in v], device=dev) for l, v in by_layer.items()}
    layer_pos = {l: torch.tensor([p for _, p in v], device=dev) for l, v in by_layer.items()}

    src_cache = torch.zeros(len(nodes), N, model.cfg.d_model, device=dev)
    with torch.no_grad():
        for s in range(0, N, cache_bs):
            sl = slice(s, min(s + cache_bs, N))
            def make_hook(layer):
                def hook(act, hook=None):
                    src_cache[layer_nodes[layer], sl] = act[:, layer_pos[layer], :].float().transpose(0, 1)
                    return act
                return hook
            with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                model.run_with_hooks(src_tok[sl],
                                     fwd_hooks=[(f"blocks.{l}.hook_resid_post", make_hook(l)) for l in by_layer],
                                     return_type=None)
    return dict(base_tok=base_tok, src_cache=src_cache, gold=gold, L=L, N=N,
                layers=sorted(by_layer), layer_nodes=layer_nodes, layer_pos=layer_pos)


def patched_logits(model, prep, idx, m, Q, amp=True):
    """Intervene (base <- source) in node subspaces, one fused op per layer. Returns [B, vocab]
    logits at the position predicting the tens digit."""
    def make_hook(layer):
        nid, pos = prep["layer_nodes"][layer], prep["layer_pos"][layer]
        def hook(act, hook=None):
            cur = act[:, pos, :].float()                                   # [B, n, d]
            src = prep["src_cache"][nid[:, None], idx[None, :]].transpose(0, 1)  # [B, n, d]
            q = Q[nid]                                                     # [n, d, k]
            coef = torch.einsum("bnd,ndk->bnk", src - cur, q)
            proj = torch.einsum("bnk,ndk->bnd", coef, q)
            new = cur + m[nid][None, :, None] * proj
            act = act.clone()
            act[:, pos, :] = new.to(act.dtype)
            return act
        return hook
    with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
        logits = model.run_with_hooks(prep["base_tok"][idx],
                                      fwd_hooks=[(f"blocks.{l}.hook_resid_post", make_hook(l)) for l in prep["layers"]],
                                      return_type="logits")
    return logits[:, prep["L"] - 1, :].float()


@torch.no_grad()
def evaluate_iia(model, prep, mask, Q, batch_size=100, amp=True):
    dev = prep["base_tok"].device
    m = torch.as_tensor(mask, device=dev).float()
    hits = []
    for s in range(0, prep["N"], batch_size):
        idx = torch.arange(s, min(s + batch_size, prep["N"]), device=dev)
        pred = patched_logits(model, prep, idx, m, Q, amp).argmax(-1)
        hits.append(pred == prep["gold"][idx])
    return torch.cat(hits).float().mean().item()


def random_mask_baseline(model, prep, n_nodes, k, Q, n_trials=20, amp=True):
    accs = []
    for _ in range(n_trials):
        m = torch.zeros(n_nodes); m[torch.randperm(n_nodes)[:k]] = 1
        accs.append(evaluate_iia(model, prep, m, Q, amp=amp))
    return float(np.mean(accs))


def leave_one_out(model, prep, mask, Q, nodes, amp=True):
    mask = torch.as_tensor(mask).clone()
    rows = []
    for i in torch.nonzero(mask).flatten().tolist():
        m = mask.clone(); m[i] = 0
        rows.append((nodes[i], evaluate_iia(model, prep, m, Q, amp=amp)))
    return rows


def summarize_nodes(sel):
    by_pos = {}
    for layer, pos in sel:
        by_pos.setdefault(pos, []).append(layer)
    lines = []
    for pos in sorted(by_pos):
        layers = sorted(by_pos[pos]); runs, start = [], layers[0]
        for a, b in zip(layers, layers[1:] + [None]):
            if b is None or b != a + 1:
                runs.append(f"{start}" if start == a else f"{start}-{a}"); start = b
        lines.append(f"  pos {pos}: layers {', '.join(runs)}")
    return "\n".join(lines) if lines else "  (no nodes selected)"


def alignment_find(model, prep, nodes, subspace_dim=1, init_logit=-0.5, lambda_=0.005, K=4.0,
                   lr_alpha=3e-2, lr_basis=1e-2, n_steps=300, batch_size=64, micro_batch=32,
                   hard=True, amp=True, seed=0, log_every=25):
    torch.manual_seed(seed)
    dev = prep["base_tok"].device
    N, n = prep["N"], len(nodes)
    sub = DASSubspaces(n, model.cfg.d_model, subspace_dim).to(dev)
    alpha = nn.Parameter(torch.full((n,), float(init_logit), device=dev))
    opt = torch.optim.Adam([{"params": [alpha], "lr": lr_alpha},
                            {"params": sub.parameters(), "lr": lr_basis}])
    t0 = time.time()
    pbar = tqdm(range(n_steps))
    for step in pbar:
        opt.zero_grad(set_to_none=True)
        perm = torch.randperm(N, device=dev)[:batch_size]
        mask_used, mask_soft = binary_concrete(alpha, K, hard)
        m_leaf = mask_used.detach().clone().requires_grad_(True)
        Q = sub.basis()
        Q_leaf = Q.detach().clone().requires_grad_(True)
        ce_total = 0.0
        for s in range(0, len(perm), micro_batch):
            idx = perm[s:s + micro_batch]
            logits = patched_logits(model, prep, idx, m_leaf, Q_leaf, amp)
            loss = F.cross_entropy(logits, prep["gold"][idx], reduction="sum") / len(perm)
            loss.backward()
            ce_total += loss.item()
        if not math.isfinite(ce_total):
            raise RuntimeError("Non-finite loss (fp16 overflow?). Re-run with amp=False.")
        Q.backward(Q_leaf.grad)                           # one backward through QR
        probs = torch.sigmoid(K * alpha)
        ((mask_used * m_leaf.grad).sum() + lambda_ * probs.sum()).backward()
        opt.step()
        if step == 2:
            print(f"~{(time.time() - t0) / 3:.1f}s/step -> ~{(time.time() - t0) / 3 * n_steps / 60:.1f} min total")
        if step % log_every == 0 or step == n_steps - 1:
            pbar.set_postfix(ce=f"{ce_total:.3f}", sel=int((alpha > 0).sum()))
    with torch.no_grad():
        hard_mask = (alpha > 0).long().cpu()
        Qf = sub.basis().detach()
    return dict(alpha=alpha.detach().cpu(), mask_hard=hard_mask, Q=Qf,
                selected_nodes=[nodes[i] for i in range(n) if hard_mask[i]])

In [ ]:
def run_carry_search(model, nodes, n_samples=300, seed=0, amp=True, do_loo=True,
                     n_random_trials=20, **search_kwargs):
    T = time.time()
    random.seed(seed); torch.manual_seed(seed)
    check_tokenization(model)
    digit_ids = get_digit_ids(model)
    data = create_filtered_dataset(model, digit_ids, n_samples=n_samples, seed=seed, amp=amp)
    gold_ids = [digit_ids[r[3]] for r in data]            # score against SOURCE tens digit
    prep = prepare(model, data, nodes, gold_ids, amp=amp)
    print(f"data+cache ready in {time.time() - T:.0f}s")

    Q0 = DASSubspaces(len(nodes), model.cfg.d_model, search_kwargs.get("subspace_dim", 1)).to(device).basis().detach()
    print("no-intervention acc:", evaluate_iia(model, prep, torch.zeros(len(nodes)), Q0, amp=amp))

    res = alignment_find(model, prep, nodes, amp=amp, seed=seed, **search_kwargs)
    k = int(res["mask_hard"].sum())
    print(f"selected {k}/{len(nodes)} nodes\n{summarize_nodes(res['selected_nodes'])}")
    found = evaluate_iia(model, prep, res["mask_hard"], res["Q"], amp=amp)
    print(f"FOUND mask IIA (source tens acc, train pairs): {found:.3f}")
    rand = loo = None
    if k > 0:
        rand = random_mask_baseline(model, prep, len(nodes), k, res["Q"], n_random_trials, amp)
        print(f"RANDOM same-size mask IIA (mean of {n_random_trials}): {rand:.3f}")
        if do_loo and k <= 40:
            loo = leave_one_out(model, prep, res["mask_hard"], res["Q"], nodes, amp)
            for node, v in sorted(loo, key=lambda x: x[1]):
                print(f"  without {node}: {v:.3f}")
    print(f"done in {time.time() - T:.0f}s")
    return dict(result=res, found=found, random=rand, loo=loo, data=data, prep=prep)

In [ ]:
out = run_carry_search(
    model, nodes,
    n_samples=300,
    subspace_dim=1,
    lambda_=0.005, K=4.0,
    n_steps=300, batch_size=64, micro_batch=32,   # raise micro_batch to 64 if memory allows
    lr_alpha=3e-2, lr_basis=1e-2,
    amp=True,                                      # set False if you see non-finite loss
)
selected_nodes = out["result"]["selected_nodes"]
selected_nodes

**Notes**
- If the first printed `s/step` is still > ~3 s, lower `batch_size` or restrict `nodes` (e.g. fewer layers / positions 8-10).
- Caveat inherited from the original method: a node whose hard mask is 0 gets *zero* gradient for its basis `Q` (the intervention is multiplied by 0), so unselected nodes keep their random basis and are scored with it. If selection looks unstable, try `hard=False` for the first ~50 steps (soft masks train every `Q`), then switch to hard.
- Reported IIA is on the optimization pairs; generate a separate screened set (different `seed`) for generalization.